In [85]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device

'cpu'

In [43]:
data = pd.read_csv('../Shared-the1000/structuralparameters-vs-capacities.csv', index_col=False, header=0)
data

,usablegc,usablevc,density,porosity,Ri,ssa,specific
0,0.015114,0.000241,1.596496,0.007,1.61,211,0.004
1,0.040332,0.000522,1.294051,0.007,1.70,361,0.005
2,0.051645,0.001990,3.851133,0.036,2.02,271,0.009
3,0.052998,0.001876,3.537582,0.014,1.86,232,0.004
4,0.053928,0.000946,1.752954,0.021,1.66,663,0.012
...,...,...,...,...,...,...,...
995,4.350774,0.015634,0.343634,0.675,13.05,4726,1.963
996,4.412086,0.015546,0.336735,0.663,11.24,4885,1.968
997,5.115723,0.015837,0.293674,0.707,8.13,5126,2.406
998,8.004842,0.016169,0.185792,0.790,10.77,5894,4.249


In [44]:
limites_tabla = pd.read_csv('../Shared-the1000/limits.csv', index_col=False, header=0)
limites_tabla

,Variable,Units,lowlimit,highlimit
0,usablegc,wt.%,0.000001,9.00
1,usablevc,kg/L,0.000001,0.05
2,density,kg/L,0.100000,5.00
3,porosity,dimensionless,0.001000,0.90
4,Ri,angstroms,1.000000,20.00
5,ssa,m^2/g,100.000000,10000.00
6,specific,cm^3/g,0.001000,10.00


In [45]:
limit_inputs = np.array(limites_tabla.iloc[0:2, -2:]).T
limit_inputs

array([[1.e-06, 1.e-06],
       [9.e+00, 5.e-02]])

In [46]:
limit_outputs = np.array(limites_tabla.iloc[2:, -2:]).T
limit_outputs

array([[1.e-01, 1.e-03, 1.e+00, 1.e+02, 1.e-03],
       [5.e+00, 9.e-01, 2.e+01, 1.e+04, 1.e+01]])

In [47]:
escaler_X = MinMaxScaler().fit(limit_inputs)
escaler_y = MinMaxScaler().fit(limit_outputs)

In [48]:
X = escaler_X.transform(np.array(data.iloc[:, :2]))

In [49]:
y = escaler_y.transform(np.array(data.iloc[:, 2:]))

In [50]:
idx_train, idx_test = train_test_split(np.arange(1000), test_size=1/3, random_state=42)

In [51]:
def evaluate_with_no_torchmetrics(modelo, X, y):
    """
    Evaluación NO  usando torchmetrics (recomendado para producción)
    """
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error
    
    # Inicializar métricas
    y_predict = modelo.predict(X=X)
    
    results = {
        'mae': mean_absolute_error(y_pred=y_predict, y_true=y),
        'mse': mean_squared_error(y_pred=y_predict, y_true=y),
        'rmse': root_mean_squared_error(y_pred=y_predict, y_true=y),
        'r2': r2_score(y_pred=y_predict, y_true=y, multioutput='uniform_average'),
        'mape': mean_absolute_percentage_error(y_pred=y_predict, y_true=y)
    }
    
    return results

In [107]:
class SVR_regressor:
    def __init__(self, n_outputs=5):
        # Constructor
        self.n_outputs = n_outputs
        self.regressor = [LinearSVR(max_iter=100000) for i in range(n_outputs)]
    
    def fit(self, X, y):
        for i in range(self.n_outputs):
            self.regressor[i].fit(X, y[:, i])

    def predict(self, X):
        y = np.zeros((X.shape[0], self.n_outputs), dtype=float)
        for i in range(self.n_outputs):
            y[:, i] = self.regressor[i].predict(X)
        return y

In [108]:
regressor_svm = SVR_regressor(n_outputs=5)

In [109]:
regressor_svm.fit(X[idx_train], y[idx_train])

In [110]:
evaluate_with_no_torchmetrics(modelo=regressor_svm, X=X[idx_test], y=y[idx_test])

{'mae': 0.027697070812279612,
 'mse': 0.002352472380220028,
 'rmse': 0.04371536855825432,
 'r2': 0.7633236968687493,
 'mape': 0.3559980336412829}

In [111]:
class TREE_regressor:
    def __init__(self, n_outputs=5):
        # Constructor
        self.n_outputs = n_outputs
        self.regressor = [DecisionTreeRegressor() for i in range(n_outputs)]
    
    def fit(self, X, y):
        for i in range(self.n_outputs):
            self.regressor[i].fit(X, y[:, i])

    def predict(self, X):
        y = np.zeros((X.shape[0], self.n_outputs), dtype=float)
        for i in range(self.n_outputs):
            y[:, i] = self.regressor[i].predict(X)
        return y

In [112]:
regressor_tree = TREE_regressor(n_outputs=5)

In [113]:
regressor_tree.fit(X[idx_train], y[idx_train])

In [114]:
evaluate_with_no_torchmetrics(modelo=regressor_tree, X=X[idx_train], y=y[idx_train])

{'mae': 5.938692980821501e-20,
 'mse': 2.8303362358393498e-36,
 'rmse': 7.975024993269976e-19,
 'r2': 1.0,
 'mape': 3.797460014424435e-19}

In [115]:
evaluate_with_no_torchmetrics(modelo=regressor_tree, X=X[idx_test], y=y[idx_test])

{'mae': 0.022907817427212436,
 'mse': 0.0017451963760451415,
 'rmse': 0.03706893062129669,
 'r2': 0.8290769831307866,
 'mape': 0.27977228868638904}